In [0]:
from pyspark.sql.functions import col, when, current_timestamp, concat, lit

print("🛠️ Commencing Silver Layer transformations with corrected column names...")

# =========================================================================
# 1. READ FROM YOUR FRESH BRONZE LAYER TABLES
# =========================================================================
bronze_telematics = spark.read.table("claim_investigation_analysis.01_bronze.telematics_raw_batch")
bronze_notes = spark.read.table("claim_investigation_analysis.01_bronze.claim_notes_raw_batch")

# =========================================================================
# 2. TRANSFORM BLOCK A: Clean and Structure Your Telematics Data (Corrected Schema)
# =========================================================================
silver_telematics_cleaned = (bronze_telematics
    .filter(col("trip_distance") > 0) # Dropping zero-distance error rows
    .select(
        col("pickup_zip").alias("device_id"),              # Swapped to pickup_zip as device proxy
        col("tpep_pickup_datetime").alias("trip_start_time"), # Corrected NYC Taxi schema name
        col("tpep_dropoff_datetime").alias("trip_end_time"),   # Corrected NYC Taxi schema name
        col("trip_distance").cast("double"),
        col("fare_amount").cast("double").alias("cost_impact"),
        current_timestamp().alias("processed_at")
    )
)

# Write out the clean structured metrics to your 02_silver schema
(silver_telematics_cleaned.write
    .format("delta")
    .mode("overwrite") 
    .saveAsTable("claim_investigation_analysis.02_silver.telematics_cleaned"))

print("✅ Telematics data polished and committed to 02_silver.telematics_cleaned!")

# =========================================================================
# 3. TRANSFORM BLOCK B: AI Text Data Engineering for your RAG Model
# =========================================================================
silver_notes_for_rag = (bronze_notes
    .filter(col("c_comment").isNotNull()) 
    .select(
        col("c_custkey").cast("string").alias("claim_id"),
        col("c_name").alias("account_holder"),
        col("c_mktsegment").alias("risk_classification"),
        # Text Engineering: Merging fields into a complete sentence layout for embedding models
        concat(
            lit("Claim ID: "), col("c_custkey"), 
            lit(" belongs to policy holder "), col("c_name"), 
            lit(". Risk category segment is classified as "), col("c_mktsegment"), 
            lit(". Official investigator field adjustor notes statement reads: '"), col("c_comment"), lit("'")
        ).alias("ai_text_chunk"),
        current_timestamp().alias("processed_at")
    )
)

# Write out the text assets to your 02_silver schema
(silver_notes_for_rag.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("claim_investigation_analysis.02_silver.claim_notes_cleaned"))

print("🧠 Text vector candidate assets engineered and committed to 02_silver.claim_notes_cleaned!")
